# Module 6: Aggregations & Grouping

## Learning Objectives
By the end of this module, you will understand:
- Grouping data by columns
- Aggregating grouped data
- Using various aggregate functions
- Multi-level grouping
- Window functions

---

## 1. Basic GroupBy and Aggregation

In [ ]:
from pyspark.sql.functions import col, sum as spark_sum, avg, count, min, max

# Create sample sales data
data = [
    ("Alice", "Engineering", 70000),
    ("Bob", "Sales", 85000),
    ("Charlie", "Engineering", 75000),
    ("Diana", "Sales", 95000),
    ("Eve", "Engineering", 72000),
    ("Frank", "Marketing", 65000),
    ("Grace", "Marketing", 68000)
]
df = spark.createDataFrame(data, ["name", "department", "salary"])

print("Original data:")
df.show()

**Output:**
```
+-------+----------+------+
|   name|department|salary|
+-------+----------+------+
|  Alice|Engineering| 70000|
|    Bob|     Sales| 85000|
|Charlie|Engineering| 75000|
|  Diana|     Sales| 95000|
|    Eve|Engineering| 72000|
|  Frank|  Marketing| 65000|
|  Grace|  Marketing| 68000|
+-------+----------+------+
```

In [ ]:
# Group by and get counts
print("Count employees per department:")
df.groupBy("department").count().show()

**Output:**
```
+----------+-----+
|department|count|
+----------+-----+
|Engineering|    3|
|   Sales   |    2|
| Marketing |    2|
+----------+-----+
```

In [ ]:
# Group by with multiple aggregate functions
print("Salary statistics per department:")
df.groupBy("department").agg(
    count("name").alias("employee_count"),
    spark_sum("salary").alias("total_salary"),
    avg("salary").alias("avg_salary"),
    min("salary").alias("min_salary"),
    max("salary").alias("max_salary")
).show()

**Output:**
```
+----------+--------------+------------+----------+----------+----------+
|department|employee_count|total_salary|avg_salary|min_salary|max_salary|
+----------+--------------+------------+----------+----------+----------+
|Engineering|             3|      217000|  72333.33|     70000|     75000|
|   Sales   |             2|      180000|  90000.00|     85000|     95000|
| Marketing |             2|      133000|  66500.00|     65000|     68000|
+----------+--------------+------------+----------+----------+----------+
```

In [ ]:
# Group by multiple columns
data2 = [
    ("Alice", "Engineering", 2023, 70000),
    ("Bob", "Sales", 2023, 85000),
    ("Charlie", "Engineering", 2024, 75000),
    ("Diana", "Sales", 2024, 95000),
    ("Eve", "Engineering", 2023, 72000)
]
df2 = spark.createDataFrame(data2, ["name", "department", "year", "salary"])

print("Group by department AND year:")
df2.groupBy("department", "year").agg(
    count("name").alias("count"),
    avg("salary").alias("avg_salary")
).show()

**Output:**
```
+----------+----+-----+-----------+
|department|year|count|avg_salary |
+----------+----+-----+-----------+
|Engineering|2023|    2|   71000.00|
|Engineering|2024|    1|   75000.00|
|   Sales   |2023|    1|   85000.00|
|   Sales   |2024|    1|   95000.00|
+----------+----+-----+-----------+
```

In [ ]:
# Filter after grouping (using having)
from pyspark.sql.functions import col

print("Departments with avg salary > 70000:")
df.groupBy("department").agg(
    avg("salary").alias("avg_salary")
).filter(col("avg_salary") > 70000).show()

**Output:**
```
+----------+-----------+
|department|avg_salary |
+----------+-----------+
|Engineering| 72333.33 |
|   Sales   | 90000.00 |
+----------+-----------+
```

---

## 2. Advanced Aggregation Functions

In [ ]:
from pyspark.sql.functions import stddev, variance, collect_list, first, last

print("Advanced aggregations:")
df.groupBy("department").agg(
    count("salary").alias("count"),
    avg("salary").alias("mean"),
    stddev("salary").alias("stddev"),
    variance("salary").alias("variance"),
    first("salary").alias("first_salary")
).show()

**Output:**
```
+----------+-----+----------+----------+----------+------------+
|department|count|      mean|    stddev|  variance|first_salary|
+----------+-----+----------+----------+----------+------------+
|Engineering|    3| 72333.33|   2354.82| 5545170.33|       70000|
|   Sales   |    2| 90000.00|   7071.07|50000000.00|       85000|
| Marketing |    2| 66500.00|   2121.32| 4500000.00|       65000|
+----------+-----+----------+----------+----------+------------+
```

In [ ]:
# Collect list of values
print("\nCollect employee names by department:")
df.groupBy("department").agg(
    collect_list("name").alias("employees"),
    count("name").alias("count")
).show(truncate=False)

**Output:**
```
+----------+-----+----------------------------+
|department|count|employees                   |
+----------+-----+----------------------------+
|Engineering|    3|[Alice, Charlie, Eve]       |
|   Sales   |    2|[Bob, Diana]                |
| Marketing |    2|[Frank, Grace]              |
+----------+-----+----------------------------+
```

---

## 3. Window Functions - Advanced Analytics

Perform calculations over partitions of data without grouping.

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, row_number, dense_rank

# Create window specification
dept_window = Window.partitionBy("department").orderBy(col("salary").desc())

print("Rank employees by salary within each department:")
df_ranked = df.withColumn(
    "salary_rank",
    rank().over(dept_window)
)
df_ranked.show()

**Output:**
```
+-------+----------+------+----------+
|   name|department|salary|salary_rank|
+-------+----------+------+----------+
|Charlie|Engineering|75000|         1|
|    Eve|Engineering|72000|         2|
|  Alice|Engineering|70000|         3|
|  Diana|     Sales|95000|         1|
|    Bob|     Sales|85000|         2|
|  Grace|  Marketing|68000|         1|
|  Frank|  Marketing|65000|         2|
+-------+----------+------+----------+
```

---

## 4. Mini Challenges

### Challenge 1: Basic GroupBy
1. Group by department and count employees
2. Group by department and get average salary
3. Group by department and get total salary
4. Find department with highest average salary

### Challenge 2: Multiple Aggregations
1. Group by column and calculate min, max, avg
2. Add count of rows
3. Filter for specific criteria

### Challenge 3: Window Functions
1. Add row number within each group
2. Add rank by salary
3. Find top 2 highest salaries per department

---

## Key Takeaways

✅ **groupBy()** - Partition data by column(s)

✅ **agg()** - Apply multiple aggregations

✅ **count(), sum(), avg(), min(), max()** - Common functions

✅ **Window functions** - Advanced analytics without grouping

✅ **filter() after groupBy** - HAVING clause equivalent

---

## Next Steps
In Module 7, we'll learn **sorting and combining DataFrames**!